In [33]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from keras.datasets import fashion_mnist

In [34]:
%load_ext autoreload
%autoreload 2
from Model import NeuralNet, InputLayer, DenseLayer, Sigmoid, Tanh, ReLU, Softmax, OneHotEncoder, MinMaxScaler

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [35]:
import wandb

In [36]:
# Data loading and preprocessing
def load_and_preprocess_data():
    print("Loading Fashion MNIST data...")
    (train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()
    
    # Split validation set
    train_images, val_images, train_labels, val_labels = train_test_split(
        train_images, train_labels, test_size=0.2, random_state=42
    )

    # Take small portions of the dataset
    SUBSET_SIZES = {
        'train': 2000,
        'val': 500,
        'test': 250
    }

    # Select subsets
    train_images = train_images[:SUBSET_SIZES['train']]
    train_labels = train_labels[:SUBSET_SIZES['train']]
    
    val_images = val_images[:SUBSET_SIZES['val']]
    val_labels = val_labels[:SUBSET_SIZES['val']]
    
    test_images = test_images[:SUBSET_SIZES['test']]
    test_labels = test_labels[:SUBSET_SIZES['test']]

    # Reshape and scale data
    def process_images(images, scaler=None):
        # Flatten images to (num_samples, 784)
        flattened = images.reshape(images.shape[0], -1)
        
        # Scale using MinMaxScaler
        if scaler is None:
            scaler = MinMaxScaler()
            scaled = scaler.fit_transform(flattened)
            return scaled.T, scaler  # Return transposed data and scaler for validation/test
        else:
            return scaler.transform(flattened).T  # Return transposed data

    # Fit scaler on training data
    train_features, scaler = process_images(train_images)
    
    # Transform validation and test data
    val_features = process_images(val_images, scaler)
    test_features = process_images(test_images, scaler)

    return (
        train_features,
        val_features,
        test_features,
        train_labels,
        val_labels,
        test_labels
    )

In [37]:
# Activation function mapper
def get_activation(activation_name):
    return {
        'Sigmoid': Sigmoid(),
        'Tanh': Tanh(),
        'ReLU': ReLU()
    }[activation_name]

In [38]:
# Training and evaluation
def train_and_evaluate(config=None):
    with wandb.init(config=config):
        config = wandb.config
        
        # Load and prepare data
        X_train, X_val, X_test, y_train, y_val, y_test = load_and_preprocess_data()
        
        # Encode labels
        encoder = OneHotEncoder()
        train_targets = encoder.fit_transform(y_train, 10)
        val_targets = encoder.transform(y_val)
        test_targets = encoder.transform(y_test)

        # Create network
        layers = [
            InputLayer(data=X_train),
            DenseLayer(units=config.size_hidden_layer, 
                      activation=get_activation(config.activation), 
                      name="Hidden"),
            DenseLayer(units=10, activation=Softmax(), name="Output")
        ]
        
        # Initialize model
        model = NeuralNet(
            layers=layers,
            batch_size=config.batch_size,
            optimizer_name=config.optimizer,
            init_method=config.weight_init,
            epochs=config.num_epochs,
            targets=train_targets,
            loss_type=config.loss,
            X_val=X_val,
            targets_val=val_targets,
            use_wandb=True
        )
        
        # Training
        training_history = model.backward_pass()
        
        # Evaluation
        val_acc, val_loss, _ = model.evaluate(X_val, val_targets)
        test_acc, test_loss, _ = model.evaluate(X_test, test_targets)
        
        # Log metrics
        wandb.log({
            "val_accuracy": val_acc / val_targets.shape[1],
            "val_loss": val_loss,
            "test_accuracy": test_acc / test_targets.shape[1],
            "test_loss": test_loss
        })

# Sweep configuration
sweep_config = {
    "name": "complete-sweep",
    "method": "grid",
    "metric": {"name": "val_loss", "goal": "minimize"},
    "parameters": {
        "num_epochs": {"values": [10, 50]},
        "size_hidden_layer": {"values": [32, 64, 128]},
        "optimizer": {"values": ["Basic"]},
        "batch_size": {"values": [128, 1024]},
        "weight_init": {"values": ["RandomNormal", "XavierUniform"]},
        "activation": {"values": ["Sigmoid", "Tanh", "ReLU"]},
        "loss": {"values": ["CrossEntropy"]}
    }
}

In [39]:
def run_experiment():
    sweep_id = wandb.sweep(sweep_config, project="fashion-mnist-classification")
    wandb.agent(sweep_id, function=train_and_evaluate)

In [40]:
if __name__ == "__main__":
    run_experiment()

Create sweep with ID: sjzg0atq
Sweep URL: https://wandb.ai/mrsagarbiswas-iit-madras/fashion-mnist-classification/sweeps/sjzg0atq


wandb: Agent Starting Run: di8pcjkz with config:
wandb: 	activation: Sigmoid
wandb: 	batch_size: 128
wandb: 	loss: CrossEntropy
wandb: 	num_epochs: 10
wandb: 	optimizer: Basic
wandb: 	size_hidden_layer: 32
wandb: 	weight_init: RandomNormal


Loading Fashion MNIST data...


100%|██████████████████████████████████████████████████████████████████████████████████| 10/10 [00:01<00:00,  6.13it/s]


epoch,▁▂▃▃▄▅▆▆▇█
test_accuracy,▁
test_loss,▁
train_accuracy,██▇▇▇▅▅▂▁▁
train_loss,▂▁▁▁▂▄▄▆██
val_accuracy,██▇▇▇▄▄▃▁▁▁
val_loss,▁▁▁▁▁▁▁▁▁▁█
epoch,9
test_accuracy,0.092
test_loss,589.7286
train_accuracy,0.1085


...

wandb: Agent Starting Run: bqjtepbk with config:
wandb: 	activation: ReLU
wandb: 	batch_size: 1024
wandb: 	loss: CrossEntropy
wandb: 	num_epochs: 50
wandb: 	optimizer: Basic
wandb: 	size_hidden_layer: 128
wandb: 	weight_init: XavierUniform


Loading Fashion MNIST data...


100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [00:02<00:00, 16.99it/s]


epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
test_accuracy,▁
test_loss,▁
train_accuracy,███▇▇▇▇▆▆▆▅▅▅▅▅▄▄▄▄▄▄▄▃▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_loss,▁▁▁▁▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▆▇▇████████████████
val_accuracy,███▇▇▇▇▇▆▆▆▆▆▆▅▅▅▅▅▄▄▄▃▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█
epoch,49
test_accuracy,0.076
test_loss,596.28752
train_accuracy,0.0945


wandb: Sweep Agent: Waiting for job.
wandb: Sweep Agent: Exiting.
